# 05 — Train + register 4 models to Unity Catalog

RF and GBT classifiers, each in a **pre-departure** (no `dep_delay`) and **in-flight**
(with `dep_delay`) variant. Hyperopt TPE with bounded search — SparkML on serverless
caps model size at 100 MB. All runs log to MLflow. Champions are registered under
3-level UC names with the `@champion` alias.

**Exit test** (Phase 2 of MIGRATION_PLAN.md): re-open the model by alias from a fresh
session and score 10 rows. That's the money shot.

## Environment check
**Environment version 4** is required for `pyspark.ml` and `mlflow.spark` on serverless.

In [ ]:
import sys
sys.path.append("..")

import mlflow
import pyspark.ml
from src import config

print(f"Python:   {sys.version.split()[0]}")
print(f"MLflow:   {mlflow.__version__}")
print(f"PySpark:  {pyspark.__version__}")

## MLflow registry setup — the actual Phase 2 fix
The original project set the workspace registry (`databricks`) at train time and read
from Unity Catalog (`databricks-uc`) at score time — an unresolvable name mismatch.

In [ ]:
mlflow.set_registry_uri(config.MLFLOW_REGISTRY_URI)
mlflow.set_experiment(config.MLFLOW_EXPERIMENT)
print(f"Registry: {config.MLFLOW_REGISTRY_URI}")
print(f"Experiment: {config.MLFLOW_EXPERIMENT}")

## Load Gold, split

In [ ]:
from pyspark.ml.feature import VectorAssembler, VectorSlicer
from pyspark.sql.functions import col

gold = spark.table(config.GOLD).select("features", "label", "dep_delay")
train, test = gold.randomSplit(
    [config.TRAIN_FRACTION, 1.0 - config.TRAIN_FRACTION],
    seed=config.RANDOM_SEED,
)
print(f"Train: {train.count():,}  Test: {test.count():,}")

## Two feature views — the "no dep_delay" ablation

In [ ]:
# `dep_delay` is at position 11 of the numerical block (see 04_gold assembled_cols).
# The pre-departure model must not see it. We slice at position 11 for the pre-departure
# view; both views share the same StandardScaler stats.
from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

# Explode the scaled feature vector once and rebuild two views.
gold_arrays = gold.withColumn("f_arr", vector_to_array(col("features")))
n_features = len(gold_arrays.select("f_arr").first()["f_arr"])

pre_indices = [i for i in range(n_features) if i != 11]
in_indices = list(range(n_features))

In [ ]:
def with_view(df, indices, name):
    slicer = VectorSlicer(inputCol="features", outputCol=name, indices=indices)
    return slicer.transform(df).select(F.col(name).alias("features"), "label")

train_pre = with_view(train, pre_indices, "features_pre")
test_pre = with_view(test, pre_indices, "features_pre")
train_in = with_view(train, in_indices, "features_in")
test_in = with_view(test, in_indices, "features_in")

## Training loop
Bounded search space respects the 100 MB serverless SparkML cap.

In [ ]:
from hyperopt import fmin, hp, tpe, Trials, STATUS_OK
from pyspark.ml.classification import GBTClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")

RF_SPACE = {
    "numTrees": hp.choice("numTrees", [30, 40, 50]),
    "maxDepth": hp.choice("maxDepth", [6, 7, 8]),
    "minInstancesPerNode": hp.choice("minInstancesPerNode", [25, 50]),
}
GBT_SPACE = {
    "maxIter": hp.choice("maxIter", [20, 25, 30]),
    "maxDepth": hp.choice("maxDepth", [4, 5, 6]),
    "stepSize": hp.uniform("stepSize", 0.05, 0.15),
}

def _train_rf(params, tr):
    return RandomForestClassifier(
        featuresCol="features", labelCol="label",
        numTrees=int(params["numTrees"]),
        maxDepth=int(params["maxDepth"]),
        minInstancesPerNode=int(params["minInstancesPerNode"]),
        seed=config.RANDOM_SEED,
    ).fit(tr)

def _train_gbt(params, tr):
    return GBTClassifier(
        featuresCol="features", labelCol="label",
        maxIter=int(params["maxIter"]),
        maxDepth=int(params["maxDepth"]),
        stepSize=float(params["stepSize"]),
        seed=config.RANDOM_SEED,
    ).fit(tr)

def _log_metrics(model, tr, te, params, tag):
    train_auc = evaluator.evaluate(model.transform(tr))
    test_auc = evaluator.evaluate(model.transform(te))
    mlflow.log_params({f"{tag}_{k}": v for k, v in params.items()})
    mlflow.log_metric(f"{tag}_train_auc", train_auc)
    mlflow.log_metric(f"{tag}_test_auc", test_auc)
    return test_auc, model

def _search(space, train_fn, tr, te, tag, evals):
    trials = Trials()
    best_state = {"auc": -1.0, "model": None, "params": None}

    def objective(params):
        model = train_fn(params, tr)
        auc = evaluator.evaluate(model.transform(te))
        if auc > best_state["auc"]:
            best_state.update({"auc": auc, "model": model, "params": params})
        return {"loss": -auc, "status": STATUS_OK}

    fmin(objective, space, algo=tpe.suggest, max_evals=evals, trials=trials,
         rstate=None, show_progressbar=False)
    return best_state

## Register champions to Unity Catalog

In [ ]:
from mlflow.tracking import MlflowClient

def _register_champion(model, view_train_df, uc_name, run_name):
    input_example = view_train_df.limit(2).toPandas()
    with mlflow.start_run(run_name=run_name):
        mlflow.spark.log_model(
            spark_model=model,
            name="model",
            registered_model_name=uc_name,
            input_example=input_example,
        )
    client = MlflowClient()
    latest = client.get_registered_model(uc_name).latest_versions[0].version
    client.set_registered_model_alias(uc_name, config.CHAMPION_ALIAS, latest)
    print(f"{uc_name}  v{latest}  aliased @{config.CHAMPION_ALIAS}")

## RF pre-departure

In [ ]:
best = _search(RF_SPACE, _train_rf, train_pre, test_pre, "rf_pre", config.HYPEROPT_MAX_EVALS)
print(f"RF pre-departure  test AUC = {best['auc']:.4f}")
_register_champion(best["model"], train_pre, config.MODEL_RF_PRE, "rf_pre_departure")

## GBT pre-departure

In [ ]:
best = _search(GBT_SPACE, _train_gbt, train_pre, test_pre, "gbt_pre", config.HYPEROPT_MAX_EVALS)
print(f"GBT pre-departure  test AUC = {best['auc']:.4f}")
_register_champion(best["model"], train_pre, config.MODEL_GBT_PRE, "gbt_pre_departure")

## RF in-flight

In [ ]:
best = _search(RF_SPACE, _train_rf, train_in, test_in, "rf_in", config.HYPEROPT_MAX_EVALS)
print(f"RF in-flight  test AUC = {best['auc']:.4f}")
_register_champion(best["model"], train_in, config.MODEL_RF_IN, "rf_in_flight")

## GBT in-flight

In [ ]:
best = _search(GBT_SPACE, _train_gbt, train_in, test_in, "gbt_in", config.HYPEROPT_MAX_EVALS)
print(f"GBT in-flight  test AUC = {best['auc']:.4f}")
_register_champion(best["model"], train_in, config.MODEL_GBT_IN, "gbt_in_flight")

## Phase 2 exit test — load champion by alias, score 10 rows
If this cell prints predictions, the original defect is dead. Screenshot it.

In [ ]:
reloaded = mlflow.spark.load_model(f"models:/{config.MODEL_GBT_PRE}@{config.CHAMPION_ALIAS}")
reloaded.transform(test_pre.limit(10)).select("label", "prediction", "probability").show(truncate=False)